# 03 - Feature Engineering and Preprocessing

A target-free, leakage-safe preparation workflow. This notebook does not train a classifier.

## 1. Imports
Question: can engineering and preprocessing be assembled reproducibly?

In [ ]:
from pathlib import Path
import pandas as pd
from src.features.engineering import NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TitanicFeatureEngineer
from src.features.preprocessing import make_feature_pipeline, split_features_target, validate_pipeline
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent

## 2. Load processed data
Question: are the Step 01 outputs the only inputs used?

In [ ]:
train = pd.read_csv(ROOT / 'data/processed/train_clean.csv')
test = pd.read_csv(ROOT / 'data/processed/test_clean.csv')
train.shape, test.shape

## 3. Define target
Question: is Transported kept outside the model input matrix?

In [ ]:
X = train.drop(columns='Transported')
y = train['Transported'].astype(int)
{'target_in_X': 'Transported' in X.columns, 'target_rate': y.mean()}

## 4. Feature engineering
Question: can cabin, spending, and group features be derived without target information?

In [ ]:
engineer = TitanicFeatureEngineer().fit(X)
engineered = engineer.transform(X)
engineered.shape, engineered.columns.tolist()

## 5. Feature inspection
Question: what are the missingness and cardinality of final features?

In [ ]:
pd.DataFrame({'missing': engineered.isna().sum(), 'unique': engineered.nunique(dropna=True), 'dtype': engineered.dtypes.astype(str)})

## 6. Feature distributions
Question: do engineered GroupSize and HasSpending retain meaningful variation?

In [ ]:
engineered[['GroupSize', 'HasSpending', 'TotalSpending']].describe(include='all')

## 7. Final feature selection
Question: which numerical and categorical fields are sent to preprocessing?

In [ ]:
{'numerical': NUMERICAL_FEATURES, 'categorical': CATEGORICAL_FEATURES}

## 8. Train/validation split
Question: is the holdout stratified and separate from Kaggle test data?

In [ ]:
X_train, X_valid, y_train, y_valid = split_features_target(train)
{'train_rows': len(X_train), 'validation_rows': len(X_valid), 'train_target_rate': y_train.mean(), 'validation_target_rate': y_valid.mean()}

## 9. Preprocessing pipeline
Question: are imputation, scaling, and encoding contained inside one train-fit pipeline?

In [ ]:
pipeline = make_feature_pipeline()
pipeline

## 10. Leakage checks
Question: can validation and test remain excluded from fitted preprocessing statistics?

In [ ]:
result = validate_pipeline(train, test)
{'target_absent': 'Transported' not in result['X_train'], 'unseen_category_safe': result['unknown_categories_safe'], 'feature_count': result['feature_count']}

## 11. Transformation validation
Question: are transformed train, validation, and test matrices compatible and finite?

In [ ]:
result['train_shape'], result['valid_shape'], result['test_shape']

## 12. Summary
The pipeline is ready to be paired with candidate classifiers in Step 04. See `docs/FEATURE_ENGINEERING.md` for decisions, exclusions, and the leakage audit.